# SpottingSalmon
YOLO model logging script
- To register the final model as an MLFlow model

### Set up:

In [0]:
%pip install --quiet numpy==1.26.4  mlflow ultralytics

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.pyfunc
import ultralytics
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, TensorSpec, DataType, ColSpec
import numpy as np
from ultralytics import YOLO
from pyspark.sql import functions as F
import os
import cv2

### 1. Model logging

In [0]:
# Run ID and artifact relative path
run_id = "cc1ca4e121884869ac75efcba1a0af4a"
artifact_path = "weights/best.pt"

# Download to local temp dir
model_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path=artifact_path
)

print(f"Model downloaded to: {model_path}")

# Load the model
model = YOLO(model_path)

In [0]:
video_path = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"
print("Video exists?", os.path.exists(video_path))

In [0]:
cap = cv2.VideoCapture(video_path)
print("Opened?", cap.isOpened())

success, frame = cap.read()
print("First frame read:", success)
cap.release()

In [0]:

video_path = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"  # <-- update if needed
print("Video exists?", os.path.exists(video_path))

In [0]:
import mlflow.pyfunc
import pandas as pd
from ultralytics import YOLO
import cv2
import os
import shutil
import uuid
from databricks.sdk import WorkspaceClient

#################
# Single logger #
#################

def _log(msg: str):
    # Single logger (stdout) so logs don't appear twice in Databricks
    print(msg, flush=True)

###################
# UC file to temp #
###################

def download_uc_file_to_tmp(w: WorkspaceClient, uc_path: str) -> str:
    """
    Downloads a UC Volume file (e.g. /Volumes/...) to a local temp path under /tmp.
    This is needed in Model Serving because the serving container may not have direct
    filesystem access to /Volumes.
    Supports both newer and older databricks-sdk versions.
    """
    # Use a unique temp path per request to avoid overwriting in concurrent serving
    local_path = f"/tmp/{uuid.uuid4().hex}.mp4"
    _log(f"[INFO] Downloading UC file: {uc_path}")

    files_api = w.files

    # Newer SDKs provide download_to(...)
    if hasattr(files_api, "download_to"):
        files_api.download_to(uc_path, local_path)
    else:
        # Older SDKs provide download(...) returning an object with .contents (binary stream)
        resp = files_api.download(uc_path)
        stream = getattr(resp, "contents", None)
        if stream is None:
            raise RuntimeError("Files API response missing `.contents` attribute (SDK too old).")

        with stream as f, open(local_path, "wb") as out:
            shutil.copyfileobj(f, out, length=1024 * 1024)  # copy in ~1MB chunks

    # Log a quick confirmation that download really happened
    size_mb = os.path.getsize(local_path) / (1024 * 1024)
    _log(f"[INFO] Download complete -> {local_path} ({size_mb:.2f} MB)")
    return local_path


###############
# Read frames #
###############

def read_video_frames(video_path: str, w: WorkspaceClient | None = None):
    """
    Generator that yields (frame_number, frame) from the video.
    - Notebook mode: reads directly from the given path.
    - Serving mode: downloads the UC Volume file to /tmp first, then reads it.
    """
    _log(f"[INFO] Reading video: {video_path}")

    cleanup = False
    local_path = video_path

    # In serving, use Files API to make the video available locally
    if w is not None:
        local_path = download_uc_file_to_tmp(w, video_path)
        cleanup = True
    else:
        # In notebooks, ensure the path is actually accessible
        if not os.path.exists(local_path):
            raise FileNotFoundError(f"Video path not accessible: {local_path}")

    cap = cv2.VideoCapture(local_path)

    # Fail fast if OpenCV can't open the file (bad path, corrupted file, missing codecs, etc.)
    if not cap.isOpened():
        raise RuntimeError(f"OpenCV could not open video: {local_path}")

    # Log basic metadata to confirm the file opened correctly
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    _log(f"[INFO] Video opened OK -> {local_path} (frames={frame_count}, fps={fps:.2f})")

    frame_number = 0
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                _log(f"[INFO] End of video at frame {frame_number}")
                break

            yield frame_number, frame
            frame_number += 1
    finally:
        # Always release the capture and remove temp file in serving mode
        cap.release()
        if cleanup:
            try:
                os.remove(local_path)
                _log(f"[INFO] Temp file removed -> {local_path}")
            except Exception as e:
                _log(f"[WARN] Could not remove temp file {local_path}: {e}")

############################
# YOLO output to python df #
############################

def convert_yolo_to_dets(result):
    """
    Converts a Ultralytics YOLO Results object into a list of plain Python dicts
    (one per bounding box), which is easy to return as a pandas DataFrame.
    """
    detections = []

    # No detections for this frame
    if result is None or result.boxes is None or len(result.boxes) == 0:
        return detections

    boxes = result.boxes
    xyxy = boxes.xyxy.cpu().numpy()
    confs = boxes.conf.cpu().numpy()

    for (x1, y1, x2, y2), conf in zip(xyxy, confs):
        detections.append({
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "confidence": float(conf)
        })

    return detections

#################
# Frame tracker #
#################

class SimpleFrameGapTracker:
    """
    Assigns a persistent track ID as long as detections appear within N frames
    of each other. If frame gap exceeds `max_gap`, a new ID is created.
    """

    def __init__(self, max_gap=5):
        self.max_gap = max_gap
        self.last_frame_seen = None
        self.current_track_id = 1

    def update(self, frame_number, detections):
        """
        detections: list of dicts [{"x1","y1","x2","y2","confidence"}]
        returns: same detections but with track_id added
        """
        if len(detections) == 0:
            return []  # no detections in this frame

        # FIRST detection ever
        if self.last_frame_seen is None:
            self.last_frame_seen = frame_number

        # If the gap is too big → new track
        elif frame_number - self.last_frame_seen > self.max_gap:
            self.current_track_id += 1

        # Update last seen frame
        self.last_frame_seen = frame_number

        # Add track_id to all detections (usually only 1 in your case)
        output = []
        for d in detections:
            output.append({
                "track_id": self.current_track_id,
                **d
            })
        return output

# ------------------------------
# YOLO + SORT MLflow Model
# ------------------------------
class FishVideoDetector(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        _log("[INFO] Loading YOLO model")
        self.model = YOLO(context.artifacts["checkpoint"])

        _log("[INFO] Initializing SORT Tracker")
        self.tracker = SimpleFrameGapTracker(max_gap=5)

        host = os.environ.get("DATABRICKS_HOST")
        token = os.environ.get("DATABRICKS_TOKEN")
        self.w = WorkspaceClient(host=host, token=token) if host and token else None

    def predict(self, context, model_input: pd.DataFrame):
        if "fish" not in model_input:
            return pd.DataFrame([])

        results = []

        for idx, video_path in enumerate(model_input["fish"]):
            _log(f"[INFO] Processing video: {video_path}")

            # reset tracker per video
            self.tracker = SimpleFrameGapTracker(max_gap=5)

            try:
                video_name = os.path.basename(video_path)

                for frame_number, frame in read_video_frames(video_path, w=self.w):
                    # YOLO inference
                    out = self.model(frame, verbose=False)
                    if isinstance(out, list):
                        out = out[0]

                    # convert YOLO → dicts
                    dets = convert_yolo_to_dets(out)

                    # tracker step
                    tracked = self.tracker.update(frame_number, dets)

                    # store results
                    for t in tracked:
                        results.append({
                            "track_id": t["track_id"],
                            "video": video_name,
                            "frame": frame_number,
                            "x1": t["x1"],
                            "y1": t["y1"],
                            "x2": t["x2"],
                            "y2": t["y2"],
                            "confidence": t.get("confidence", 0.0)
                        })

            except Exception as e:
                _log(f"[ERROR] failed video {video_path}: {e}")

        return pd.DataFrame(results)

In [0]:
import pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, ColSpec

input_example = pd.DataFrame({
    "fish": ["/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"]
})


signature = ModelSignature(
    inputs=Schema([ColSpec("string", "fish")]),
    outputs=Schema([
        ColSpec("integer", "fish_id"),
        ColSpec("string", "video"),
        ColSpec("integer", "frame"),
        ColSpec("float", "x1"),
        ColSpec("float", "y1"),
        ColSpec("float", "x2"),
        ColSpec("float", "y2"),
        ColSpec("float", "confidence")
    ])
)

In [0]:
with mlflow.start_run() as run:
    run_id = run.info.run_id

    mlflow.pyfunc.log_model(
        name="model",  # 
        python_model=FishVideoDetector(),
        input_example=input_example,
        artifacts={"checkpoint": model_path},
        signature=signature
    )

print("Logged model under run:", run_id)


### Register model

In [0]:
import mlflow

mlflow.set_registry_uri("databricks")

model_uri = f"runs:/{run_id}/model"

print("About to register model from:")
print("Run ID:", run_id)
print("Model URI:", model_uri)

result = mlflow.register_model(
    model_uri=model_uri,
    name="FishVideoDetector"
)

print("\nRegistered model version:", result.version)
print("Registered from run ID:", result.run_id)

# Extra safety check
if result.run_id == run_id:
    print("Registration confirmed: correct run ID used.")
else:
    print(" WARNING: Registered run ID does NOT match expected run ID!")



In [0]:
import mlflow.pyfunc
import pandas as pd

mlflow.set_registry_uri("databricks")

model_uri = "models:/FishVideoDetector/8"
model = mlflow.pyfunc.load_model(model_uri)

input_df = pd.DataFrame({
    "fish": [
        "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"
    ]
})

preds = model.predict(input_df)
print(preds)


In [0]:
display(preds)